<a href="https://colab.research.google.com/github/AmishaSingh0408/code_generator/blob/main/code_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Install dependency (run once)
!pip install requests -q

In [ ]:
# Cell 2 — Core generator function
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "phi3"

SYSTEM_PROMPTS = {
    "python": (
        "You are an expert Python developer. "
        "Write clean, well-commented Python code for the given task. "
        "Output only the code with inline comments. No extra explanation."
    ),
    "java": (
        "You are an expert Java developer. "
        "Write clean, well-commented Java code for the given task. "
        "Output only the code with inline comments. No extra explanation."
    ),
}

def generate_code(question: str, language: str) -> None:
    """Stream code from Ollama phi3 and print to terminal."""
    lang = language.lower().strip()
    if lang not in SYSTEM_PROMPTS:
        print(f"[ERROR] Unsupported language '{language}'. Choose: python | java")
        return

    prompt = f"{SYSTEM_PROMPTS[lang]}\n\nTask: {question}\n\nCode:"

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": True
    }

    print(f"\n{'='*60}")
    print(f"  Language : {lang.upper()}")
    print(f"  Model    : {MODEL}")
    print(f"  Task     : {question}")
    print(f"{'='*60}\n")

    try:
        with requests.post(OLLAMA_URL, json=payload, stream=True, timeout=120) as resp:
            resp.raise_for_status()
            for line in resp.iter_lines():
                if line:
                    chunk = json.loads(line)
                    token = chunk.get("response", "")
                    print(token, end="", flush=True)
                    if chunk.get("done"):
                        break
        print(f"\n\n{'='*60}")
        print("  ✓ Generation complete")
        print(f"{'='*60}\n")

    except requests.exceptions.ConnectionError:
        print("[ERROR] Cannot connect to Ollama.")
        print("  → Run: ollama serve")
        print("  → Then: ollama pull phi3")
    except requests.exceptions.HTTPError as e:
        print(f"[ERROR] Ollama HTTP error: {e}")
    except Exception as e:
        print(f"[ERROR] Unexpected error: {e}")

print("✓ Generator loaded. Run the next cell to start.")

In [ ]:
# Cell 3 — Single run: customize these two variables and run this cell

LANGUAGE = "python"   # change to "java" for Java
QUESTION = "Write a function to find all prime numbers up to N using Sieve of Eratosthenes"

generate_code(QUESTION, LANGUAGE)

In [ ]:
# Cell 4 — Interactive loop: keeps asking for input until you type 'exit'

print("Code Generator — Interactive Mode")
print("Type 'exit' at any prompt to quit.\n")

while True:
    lang = input("Language (python/java): ").strip().lower()
    if lang == "exit":
        print("Goodbye.")
        break
    if lang not in ("python", "java"):
        print("  → Please enter 'python' or 'java'\n")
        continue

    question = input("Your question: ").strip()
    if question.lower() == "exit":
        print("Goodbye.")
        break
    if not question:
        print("  → Question cannot be empty\n")
        continue

    generate_code(question, lang)

    again = input("\nGenerate another? (y/n): ").strip().lower()
    if again != "y":
        print("Goodbye.")
        break
    print()